In [1]:
import polars as pl
import boto3
import os
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")
endpoint_url = os.getenv("MLFLOW_S3_ENDPOINT_URL")
bucket_name = os.getenv("INPUT_BUCKET")
bucket_key = os.getenv("INPUT_KEY")
bucket_key

'mobile_sales_data.csv'

In [4]:
s3 = boto3.client(
    "s3",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    endpoint_url=endpoint_url
)


In [5]:
obj = s3.get_object(Bucket=bucket_name, Key=bucket_key)

In [6]:
df =pl.read_csv(obj['Body'])

Filter bad data  
Remove negatives and future dates

In [7]:
df = df.filter(
    (pl.col("Price") >= 0) &
    (pl.col("Quantity Sold") >= 0) &
    (pl.col("Dispatch Date").str.strptime(pl.Date, format="%Y-%m-%d") <= datetime.now().date())
)

Parse Dates

In [8]:
df = df.with_columns([
    pl.col("Inward Date").str.strptime(pl.Date, format="%Y-%m-%d").alias("Inward Date"),
    pl.col("Dispatch Date").str.strptime(pl.Date, format="%Y-%m-%d").alias("Dispatch Date"),
])

Feature Engineering Steps

Time-based features

In [10]:
df = df.with_columns([
    pl.col("Dispatch Date").dt.year().alias("dispatch_year"),
    pl.col("Dispatch Date").dt.month().alias("dispatch_month"),
    pl.col("Dispatch Date").dt.weekday().alias("dispatch_day_of_week"),  # Monday=1, Sunday=7
    pl.col("Dispatch Date").dt.day().alias("dispatch_day"),
])
df.head()

Product,Brand,Product Code,Product Specification,Price,Inward Date,Dispatch Date,Quantity Sold,Customer Name,Customer Location,Region,Processor Specification,RAM,ROM,dispatch_year,dispatch_month,dispatch_day_of_week,dispatch_day
str,str,str,str,i64,date,date,i64,str,str,str,str,str,str,i32,i8,i8,i8
"""Mobile Phone""","""Motorola""","""88EB4558""","""Site candidate…",78570,2023-08-02,2023-08-03,6,"""William Hess""","""South Kelsey""","""Central""","""Snapdragon 7 G…","""12GB""","""128GB""",2023,8,4,3
"""Mobile Phone""","""Samsung""","""9F975B08""","""Energy special…",159826,2025-03-19,2025-03-20,5,"""Leah Copeland""","""South Todd""","""Central""","""MediaTek Dimen…","""8GB""","""256GB""",2025,3,4,20
"""Mobile Phone""","""Dell""","""14932CAE""","""Could before a…",11670,2023-10-10,2023-10-16,6,"""Nicole Gonzale…","""South Miguel""","""North""","""Apple A-Series…","""16GB""","""256GB""",2023,10,1,16
"""Mobile Phone""","""Motorola""","""E3CF42BE""","""Responsibility…",174698,2025-02-09,2025-03-20,10,"""Miranda Clayto…","""North Deanna""","""North""","""MediaTek Dimen…","""6GB""","""64GB""",2025,3,4,20
"""Mobile Phone""","""Apple""","""F54013D6""","""To say system …",51251,2023-04-03,2023-04-19,9,"""Ms. Katie Ande…","""Lake Laurenfor…","""North""","""Snapdragon 8 G…","""4GB""","""256GB""",2023,4,3,19


Derived Metrics (Inventory Turnover)

In [22]:
df = df.with_columns(
    (pl.col("Dispatch Date") - pl.col("Inward Date")).dt.total_days().alias("days_to_sell")
)


df.head()

Product,Brand,Product Code,Product Specification,Price,Inward Date,Dispatch Date,Quantity Sold,Customer Name,Customer Location,Region,Processor Specification,RAM,ROM,dispatch_year,dispatch_month,dispatch_day_of_week,dispatch_day,Revenue,avg_price_per_brand,avg_qty_per_region,brand_code,region_code,ram_code,rom_code,spec_length,days_to_sell
str,str,str,str,i64,date,date,i64,str,str,str,str,str,str,i32,i8,i8,i8,i64,f64,f64,u32,u32,u32,u32,u32,i64
"""Mobile Phone""","""Motorola""","""88EB4558""","""Site candidate…",78570,2023-08-02,2023-08-03,6,"""William Hess""","""South Kelsey""","""Central""","""Snapdragon 7 G…","""12GB""","""128GB""",2023,8,4,3,471420,102969.754125,5.534659,0,0,0,0,76,1
"""Mobile Phone""","""Samsung""","""9F975B08""","""Energy special…",159826,2025-03-19,2025-03-20,5,"""Leah Copeland""","""South Todd""","""Central""","""MediaTek Dimen…","""8GB""","""256GB""",2025,3,4,20,799130,103993.488871,5.534659,1,0,1,1,40,1
"""Mobile Phone""","""Dell""","""14932CAE""","""Could before a…",11670,2023-10-10,2023-10-16,6,"""Nicole Gonzale…","""South Miguel""","""North""","""Apple A-Series…","""16GB""","""256GB""",2023,10,1,16,70020,103043.388109,5.474654,2,1,2,1,57,6
"""Mobile Phone""","""Motorola""","""E3CF42BE""","""Responsibility…",174698,2025-02-09,2025-03-20,10,"""Miranda Clayto…","""North Deanna""","""North""","""MediaTek Dimen…","""6GB""","""64GB""",2025,3,4,20,1746980,102969.754125,5.474654,0,1,3,2,79,39
"""Mobile Phone""","""Apple""","""F54013D6""","""To say system …",51251,2023-04-03,2023-04-19,9,"""Ms. Katie Ande…","""Lake Laurenfor…","""North""","""Snapdragon 8 G…","""4GB""","""256GB""",2023,4,3,19,461259,102809.734362,5.474654,3,1,4,1,28,16


Derived Metrics (Revenue)

In [23]:
df = df.with_columns(
    (pl.col("Price") * pl.col("Quantity Sold")).alias("Revenue")
)
df.head()

Product,Brand,Product Code,Product Specification,Price,Inward Date,Dispatch Date,Quantity Sold,Customer Name,Customer Location,Region,Processor Specification,RAM,ROM,dispatch_year,dispatch_month,dispatch_day_of_week,dispatch_day,Revenue,avg_price_per_brand,avg_qty_per_region,brand_code,region_code,ram_code,rom_code,spec_length,days_to_sell
str,str,str,str,i64,date,date,i64,str,str,str,str,str,str,i32,i8,i8,i8,i64,f64,f64,u32,u32,u32,u32,u32,i64
"""Mobile Phone""","""Motorola""","""88EB4558""","""Site candidate…",78570,2023-08-02,2023-08-03,6,"""William Hess""","""South Kelsey""","""Central""","""Snapdragon 7 G…","""12GB""","""128GB""",2023,8,4,3,471420,102969.754125,5.534659,0,0,0,0,76,1
"""Mobile Phone""","""Samsung""","""9F975B08""","""Energy special…",159826,2025-03-19,2025-03-20,5,"""Leah Copeland""","""South Todd""","""Central""","""MediaTek Dimen…","""8GB""","""256GB""",2025,3,4,20,799130,103993.488871,5.534659,1,0,1,1,40,1
"""Mobile Phone""","""Dell""","""14932CAE""","""Could before a…",11670,2023-10-10,2023-10-16,6,"""Nicole Gonzale…","""South Miguel""","""North""","""Apple A-Series…","""16GB""","""256GB""",2023,10,1,16,70020,103043.388109,5.474654,2,1,2,1,57,6
"""Mobile Phone""","""Motorola""","""E3CF42BE""","""Responsibility…",174698,2025-02-09,2025-03-20,10,"""Miranda Clayto…","""North Deanna""","""North""","""MediaTek Dimen…","""6GB""","""64GB""",2025,3,4,20,1746980,102969.754125,5.474654,0,1,3,2,79,39
"""Mobile Phone""","""Apple""","""F54013D6""","""To say system …",51251,2023-04-03,2023-04-19,9,"""Ms. Katie Ande…","""Lake Laurenfor…","""North""","""Snapdragon 8 G…","""4GB""","""256GB""",2023,4,3,19,461259,102809.734362,5.474654,3,1,4,1,28,16


Aggregated Features  
1) Average Price per Brand

In [12]:
brand_avg_price = df.group_by("Brand").agg(
    pl.col("Price").mean().alias("avg_price_per_brand")
)
brand_avg_price

Brand,avg_price_per_brand
str,f64
"""Nokia""",103626.378717
"""Acer""",103494.328811
"""Realme""",100839.122705
"""Google""",103910.344376
"""OnePlus""",101711.727559
"""Apple""",102809.734362
"""HP""",101953.270884
"""Sony""",104203.905481
"""Vivo""",99928.345779


In [13]:
df = df.join(brand_avg_price, on="Brand", how="left")
df.head()

Product,Brand,Product Code,Product Specification,Price,Inward Date,Dispatch Date,Quantity Sold,Customer Name,Customer Location,Region,Processor Specification,RAM,ROM,dispatch_year,dispatch_month,dispatch_day_of_week,dispatch_day,Revenue,avg_price_per_brand
str,str,str,str,i64,date,date,i64,str,str,str,str,str,str,i32,i8,i8,i8,i64,f64
"""Mobile Phone""","""Motorola""","""88EB4558""","""Site candidate…",78570,2023-08-02,2023-08-03,6,"""William Hess""","""South Kelsey""","""Central""","""Snapdragon 7 G…","""12GB""","""128GB""",2023,8,4,3,471420,102969.754125
"""Mobile Phone""","""Samsung""","""9F975B08""","""Energy special…",159826,2025-03-19,2025-03-20,5,"""Leah Copeland""","""South Todd""","""Central""","""MediaTek Dimen…","""8GB""","""256GB""",2025,3,4,20,799130,103993.488871
"""Mobile Phone""","""Dell""","""14932CAE""","""Could before a…",11670,2023-10-10,2023-10-16,6,"""Nicole Gonzale…","""South Miguel""","""North""","""Apple A-Series…","""16GB""","""256GB""",2023,10,1,16,70020,103043.388109
"""Mobile Phone""","""Motorola""","""E3CF42BE""","""Responsibility…",174698,2025-02-09,2025-03-20,10,"""Miranda Clayto…","""North Deanna""","""North""","""MediaTek Dimen…","""6GB""","""64GB""",2025,3,4,20,1746980,102969.754125
"""Mobile Phone""","""Apple""","""F54013D6""","""To say system …",51251,2023-04-03,2023-04-19,9,"""Ms. Katie Ande…","""Lake Laurenfor…","""North""","""Snapdragon 8 G…","""4GB""","""256GB""",2023,4,3,19,461259,102809.734362


Average Quantity Sold per Region

In [15]:
region_avg_qty = df.group_by("Region").agg(
    pl.col("Quantity Sold").mean().alias("avg_qty_per_region")
)
region_avg_qty

Region,avg_qty_per_region
str,f64
"""West""",5.492646
"""Central""",5.534659
"""East""",5.562679
"""North""",5.474654
"""South""",5.489637


In [16]:
df = df.join(region_avg_qty, on="Region", how="left")
df.head()

Product,Brand,Product Code,Product Specification,Price,Inward Date,Dispatch Date,Quantity Sold,Customer Name,Customer Location,Region,Processor Specification,RAM,ROM,dispatch_year,dispatch_month,dispatch_day_of_week,dispatch_day,Revenue,avg_price_per_brand,avg_qty_per_region
str,str,str,str,i64,date,date,i64,str,str,str,str,str,str,i32,i8,i8,i8,i64,f64,f64
"""Mobile Phone""","""Motorola""","""88EB4558""","""Site candidate…",78570,2023-08-02,2023-08-03,6,"""William Hess""","""South Kelsey""","""Central""","""Snapdragon 7 G…","""12GB""","""128GB""",2023,8,4,3,471420,102969.754125,5.534659
"""Mobile Phone""","""Samsung""","""9F975B08""","""Energy special…",159826,2025-03-19,2025-03-20,5,"""Leah Copeland""","""South Todd""","""Central""","""MediaTek Dimen…","""8GB""","""256GB""",2025,3,4,20,799130,103993.488871,5.534659
"""Mobile Phone""","""Dell""","""14932CAE""","""Could before a…",11670,2023-10-10,2023-10-16,6,"""Nicole Gonzale…","""South Miguel""","""North""","""Apple A-Series…","""16GB""","""256GB""",2023,10,1,16,70020,103043.388109,5.474654
"""Mobile Phone""","""Motorola""","""E3CF42BE""","""Responsibility…",174698,2025-02-09,2025-03-20,10,"""Miranda Clayto…","""North Deanna""","""North""","""MediaTek Dimen…","""6GB""","""64GB""",2025,3,4,20,1746980,102969.754125,5.474654
"""Mobile Phone""","""Apple""","""F54013D6""","""To say system …",51251,2023-04-03,2023-04-19,9,"""Ms. Katie Ande…","""Lake Laurenfor…","""North""","""Snapdragon 8 G…","""4GB""","""256GB""",2023,4,3,19,461259,102809.734362,5.474654


Categorical Encoding (Convert text to numbers)  
Use simple label encoding for Brand, Region, RAM, ROM  
Polars has a built-in `to_physical()` for categoricals  

In [17]:
df = df.with_columns([
    pl.col("Brand").cast(pl.Categorical).to_physical().alias("brand_code"),
    pl.col("Region").cast(pl.Categorical).to_physical().alias("region_code"),
    pl.col("RAM").cast(pl.Categorical).to_physical().alias("ram_code"),
    pl.col("ROM").cast(pl.Categorical).to_physical().alias("rom_code"),
])
df.head()

Product,Brand,Product Code,Product Specification,Price,Inward Date,Dispatch Date,Quantity Sold,Customer Name,Customer Location,Region,Processor Specification,RAM,ROM,dispatch_year,dispatch_month,dispatch_day_of_week,dispatch_day,Revenue,avg_price_per_brand,avg_qty_per_region,brand_code,region_code,ram_code,rom_code
str,str,str,str,i64,date,date,i64,str,str,str,str,str,str,i32,i8,i8,i8,i64,f64,f64,u32,u32,u32,u32
"""Mobile Phone""","""Motorola""","""88EB4558""","""Site candidate…",78570,2023-08-02,2023-08-03,6,"""William Hess""","""South Kelsey""","""Central""","""Snapdragon 7 G…","""12GB""","""128GB""",2023,8,4,3,471420,102969.754125,5.534659,0,0,0,0
"""Mobile Phone""","""Samsung""","""9F975B08""","""Energy special…",159826,2025-03-19,2025-03-20,5,"""Leah Copeland""","""South Todd""","""Central""","""MediaTek Dimen…","""8GB""","""256GB""",2025,3,4,20,799130,103993.488871,5.534659,1,0,1,1
"""Mobile Phone""","""Dell""","""14932CAE""","""Could before a…",11670,2023-10-10,2023-10-16,6,"""Nicole Gonzale…","""South Miguel""","""North""","""Apple A-Series…","""16GB""","""256GB""",2023,10,1,16,70020,103043.388109,5.474654,2,1,2,1
"""Mobile Phone""","""Motorola""","""E3CF42BE""","""Responsibility…",174698,2025-02-09,2025-03-20,10,"""Miranda Clayto…","""North Deanna""","""North""","""MediaTek Dimen…","""6GB""","""64GB""",2025,3,4,20,1746980,102969.754125,5.474654,0,1,3,2
"""Mobile Phone""","""Apple""","""F54013D6""","""To say system …",51251,2023-04-03,2023-04-19,9,"""Ms. Katie Ande…","""Lake Laurenfor…","""North""","""Snapdragon 8 G…","""4GB""","""256GB""",2023,4,3,19,461259,102809.734362,5.474654,3,1,4,1


Text Feature (Length of Product Specification - quick proxy for descriptiveness)


In [18]:
df = df.with_columns(
    pl.col("Product Specification").str.len_chars().alias("spec_length")
)
df.head()

Product,Brand,Product Code,Product Specification,Price,Inward Date,Dispatch Date,Quantity Sold,Customer Name,Customer Location,Region,Processor Specification,RAM,ROM,dispatch_year,dispatch_month,dispatch_day_of_week,dispatch_day,Revenue,avg_price_per_brand,avg_qty_per_region,brand_code,region_code,ram_code,rom_code,spec_length
str,str,str,str,i64,date,date,i64,str,str,str,str,str,str,i32,i8,i8,i8,i64,f64,f64,u32,u32,u32,u32,u32
"""Mobile Phone""","""Motorola""","""88EB4558""","""Site candidate…",78570,2023-08-02,2023-08-03,6,"""William Hess""","""South Kelsey""","""Central""","""Snapdragon 7 G…","""12GB""","""128GB""",2023,8,4,3,471420,102969.754125,5.534659,0,0,0,0,76
"""Mobile Phone""","""Samsung""","""9F975B08""","""Energy special…",159826,2025-03-19,2025-03-20,5,"""Leah Copeland""","""South Todd""","""Central""","""MediaTek Dimen…","""8GB""","""256GB""",2025,3,4,20,799130,103993.488871,5.534659,1,0,1,1,40
"""Mobile Phone""","""Dell""","""14932CAE""","""Could before a…",11670,2023-10-10,2023-10-16,6,"""Nicole Gonzale…","""South Miguel""","""North""","""Apple A-Series…","""16GB""","""256GB""",2023,10,1,16,70020,103043.388109,5.474654,2,1,2,1,57
"""Mobile Phone""","""Motorola""","""E3CF42BE""","""Responsibility…",174698,2025-02-09,2025-03-20,10,"""Miranda Clayto…","""North Deanna""","""North""","""MediaTek Dimen…","""6GB""","""64GB""",2025,3,4,20,1746980,102969.754125,5.474654,0,1,3,2,79
"""Mobile Phone""","""Apple""","""F54013D6""","""To say system …",51251,2023-04-03,2023-04-19,9,"""Ms. Katie Ande…","""Lake Laurenfor…","""North""","""Snapdragon 8 G…","""4GB""","""256GB""",2023,4,3,19,461259,102809.734362,5.474654,3,1,4,1,28


Select the Final Feature Set

In [19]:
final_columns = [
    # Target variable (What we want to predict)
    "Quantity Sold",
    # Numeric/Date features
    "Price",
    "days_to_sell",
    "Revenue",
    "dispatch_year",
    "dispatch_month",
    "dispatch_day_of_week",
    "spec_length",
    # Encoded categoricals
    "brand_code",
    "region_code",
    "ram_code",
    "rom_code",
    # Aggregated features (added context)
    "avg_price_per_brand",
    "avg_qty_per_region",
]

Create the feature DataFrame

In [24]:
feature_df = df.select(final_columns)
feature_df.head(5)

Quantity Sold,Price,days_to_sell,Revenue,dispatch_year,dispatch_month,dispatch_day_of_week,spec_length,brand_code,region_code,ram_code,rom_code,avg_price_per_brand,avg_qty_per_region
i64,i64,i64,i64,i32,i8,i8,u32,u32,u32,u32,u32,f64,f64
6,78570,1,471420,2023,8,4,76,0,0,0,0,102969.754125,5.534659
5,159826,1,799130,2025,3,4,40,1,0,1,1,103993.488871,5.534659
6,11670,6,70020,2023,10,1,57,2,1,2,1,103043.388109,5.474654
10,174698,39,1746980,2025,3,4,79,0,1,3,2,102969.754125,5.474654
9,51251,16,461259,2023,4,3,28,3,1,4,1,102809.734362,5.474654


Handle Nulls

In [25]:
feature_df = feature_df.fill_null(0)

In [26]:
feature_df.shape

(24983, 14)

Save to MinIO for the Training Pipeline

In [27]:
local_feature_file = "processed_features.parquet"
feature_df.write_parquet(local_feature_file)

In [ ]:
OUTPUT_BUCKET = "processed-features"
OUTPUT_KEY = f"features_{datetime.now().strftime('%Y-%m-%d')}.parquet"

In [ ]:
s3.upload_file(local_feature_file, OUTPUT_BUCKET, OUTPUT_KEY)

In [ ]:
with open("/tmp/output_uri.txt", "w") as f:
    f.write(f"s3://{OUTPUT_BUCKET}/{OUTPUT_KEY}")